# Lab 1: Introduction to NLP - Text Representation & Tokenization

In [15]:
import nltk
import spacy
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

print('Setup Complete!')

Setup Complete!


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\akkxm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\akkxm\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\akkxm\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\akkxm\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\akkxm\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


## EXERCISE 1: Your Turn - Tokenization
Task: Tokenize the text and answer questions.

In [16]:
from nltk.tokenize import sent_tokenize, word_tokenize

text = "The COVID-19 pandemic started in 2019-2020. Dr. Johnson said, 'We're making progress!' The vaccine costs €50-€100 in the E.U. Visit https://who.int for updates."

sentences = sent_tokenize(text)
print(f"Sentences: {len(sentences)}")

tokens = word_tokenize(text)
print(f"Total tokens: {len(tokens)}")

unique_tokens = set(tokens)
print(f"Unique tokens (types): {len(unique_tokens)}")

print(f"\nURL handling:")
print([t for t in tokens if 'http' in t.lower()])

Sentences: 4
Total tokens: 32
Unique tokens (types): 28

URL handling:
['https']


## EXERCISE 2: Your Turn - Bag-of-Words (BoW)
Task: Create BoW representation for movie reviews.

In [17]:
reviews = [
    "This movie is amazing and wonderful",
    "Terrible film, waste of time",
    "Great acting but boring plot",
    "Amazing cinematography and great story"
]

vectorizer = CountVectorizer()
bow = vectorizer.fit_transform(reviews)

word_counts = bow.toarray().sum(axis=0)
vocab = vectorizer.get_feature_names_out()
most_common_idx = word_counts.argmax()
print(f"Most common word: '{vocab[most_common_idx]}' (count: {word_counts[most_common_idx]})")

print(f"Vocabulary size: {len(vocab)}")

sparsity = 100 * (1 - bow.nnz / (bow.shape[0] * bow.shape[1]))
print(f"Sparsity: {sparsity:.1f}%")

df_bow = pd.DataFrame(bow.toarray(), columns=vocab, index=[f"Review {i+1}" for i in range(len(reviews))])
print("\nBoW Matrix:")
print(df_bow)

Most common word: 'amazing' (count: 2)
Vocabulary size: 18
Sparsity: 70.8%

BoW Matrix:
          acting  amazing  and  boring  but  cinematography  film  great  is  \
Review 1       0        1    1       0    0               0     0      0   1   
Review 2       0        0    0       0    0               0     1      0   0   
Review 3       1        0    0       1    1               0     0      1   0   
Review 4       0        1    1       0    0               1     0      1   0   

          movie  of  plot  story  terrible  this  time  waste  wonderful  
Review 1      1   0     0      0         0     1     0      0          1  
Review 2      0   1     0      0         1     0     1      1          0  
Review 3      0   0     1      0         0     0     0      0          0  
Review 4      0   0     0      1         0     0     0      0          0  


## EXERCISE 3: Your Turn - TF-IDF
Task: Find the most distinctive words in each document.

In [18]:
documents = [
    "Machine learning algorithms require large datasets",
    "Deep learning is a subset of machine learning",
    "Natural language processing uses machine learning",
    "Computer vision applies deep learning techniques"
]

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(documents)
vocab_tfidf = tfidf.get_feature_names_out()

print("Top 2 distinctive words per document:")
for i in range(len(documents)):
    scores = tfidf_matrix[i].toarray().flatten()
    top_indices = scores.argsort()[-2:][::-1]
    print(f"Doc {i+1}: {documents[i]}")
    for idx in top_indices:
        print(f"  {vocab_tfidf[idx]:15s}: {scores[idx]:.3f}")

idf_scores = tfidf.idf_
max_idf_idx = idf_scores.argmax()
print(f"\nHighest IDF word: '{vocab_tfidf[max_idf_idx]}' (IDF: {idf_scores[max_idf_idx]:.3f})")
print("This word appears in the fewest documents.")

Top 2 distinctive words per document:
Doc 1: Machine learning algorithms require large datasets
  algorithms     : 0.462
  require        : 0.462
Doc 2: Deep learning is a subset of machine learning
  learning       : 0.461
  subset         : 0.442
Doc 3: Natural language processing uses machine learning
  uses           : 0.462
  processing     : 0.462
Doc 4: Computer vision applies deep learning techniques
  vision         : 0.452
  techniques     : 0.452

Highest IDF word: 'algorithms' (IDF: 1.916)
This word appears in the fewest documents.


## EXERCISE 4: Your Turn - Text Preprocessing Pipeline
Task: Build a preprocessing pipeline for sentiment analysis.

In [19]:
from nltk.stem import WordNetLemmatizer

def preprocess_sentiment(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

review = "I absolutely LOVED this movie!!! The acting was great, but the plot was terrible. Would not recommend. 2/10"

print("Original:")
print(review)

tokens = preprocess_sentiment(review)
print("\nProcessed tokens:")
print(tokens)
print(f"\nToken count: {len(tokens)}")

Original:
I absolutely LOVED this movie!!! The acting was great, but the plot was terrible. Would not recommend. 2/10

Processed tokens:
['i', 'absolutely', 'loved', 'this', 'movie', 'the', 'acting', 'wa', 'great', 'but', 'the', 'plot', 'wa', 'terrible', 'would', 'not', 'recommend']

Token count: 17


## EXERCISE 5: Stemming vs Lemmatization Comparison

In [20]:
from nltk.stem import PorterStemmer
import time

text = """The striped bats are hanging on their feet for best performance.
The studies have shown that running improves cardiovascular health.
The children were playing with better toys than before."""

tokens = word_tokenize(text.lower())
tokens = [t for t in tokens if t.isalpha()]

stemmer = PorterStemmer()
start = time.time()
stems = [stemmer.stem(t) for t in tokens]
stem_time = time.time() - start

lemmatizer = WordNetLemmatizer()
start = time.time()
lemmas = [lemmatizer.lemmatize(t) for t in tokens]
lemma_time = time.time() - start

print("Token           | Stem            | Lemma")
print("-" * 50)
for token, stem, lemma in zip(tokens[:15], stems[:15], lemmas[:15]):
    print(f"{token:15s} | {stem:15s} | {lemma}")

print(f"\nProcessing time:")
print(f"Stemming: {stem_time*1000:.2f}ms")
print(f"Lemmatization: {lemma_time*1000:.2f}ms")

if stem_time > 0:
    print(f"Speed ratio: {lemma_time/stem_time:.1f}x slower")

print(f"\nVocabulary size:")
print(f"Original: {len(set(tokens))}")
print(f"After stemming: {len(set(stems))}")
print(f"After lemmatization: {len(set(lemmas))}")

Token           | Stem            | Lemma
--------------------------------------------------
the             | the             | the
striped         | stripe          | striped
bats            | bat             | bat
are             | are             | are
hanging         | hang            | hanging
on              | on              | on
their           | their           | their
feet            | feet            | foot
for             | for             | for
best            | best            | best
performance     | perform         | performance
the             | the             | the
studies         | studi           | study
have            | have            | have
shown           | shown           | shown

Processing time:
Stemming: 1.00ms
Lemmatization: 0.00ms
Speed ratio: 0.0x slower

Vocabulary size:
Original: 27
After stemming: 27
After lemmatization: 27


## EXERCISE 6: Real-World Application - Document Similarity

In [21]:
docs_sim = [
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks with many layers",
    "Natural language processing helps computers understand human language",
    "Computer vision is about teaching computers to see and interpret images"
]

tfidf_sim = TfidfVectorizer()
tfidf_sim_matrix = tfidf_sim.fit_transform(docs_sim)

similarity_matrix = cosine_similarity(tfidf_sim_matrix)

df_sim = pd.DataFrame(
    similarity_matrix,
    index=[f"Doc {i+1}" for i in range(len(docs_sim))],
    columns=[f"Doc {i+1}" for i in range(len(docs_sim))]
)

print("Document Similarity Matrix (Cosine Similarity):\n")
print(df_sim.round(3))

print("\nMost similar document pairs:")
for i in range(len(docs_sim)):
    for j in range(i+1, len(docs_sim)):
        print(f"Doc {i+1} vs Doc {j+1}: {similarity_matrix[i][j]:.3f}")

Document Similarity Matrix (Cosine Similarity):

       Doc 1  Doc 2  Doc 3  Doc 4
Doc 1  1.000   0.09  0.000  0.078
Doc 2  0.090   1.00  0.000  0.000
Doc 3  0.000   0.00  1.000  0.063
Doc 4  0.078   0.00  0.063  1.000

Most similar document pairs:
Doc 1 vs Doc 2: 0.090
Doc 1 vs Doc 3: 0.000
Doc 1 vs Doc 4: 0.078
Doc 2 vs Doc 3: 0.000
Doc 2 vs Doc 4: 0.000
Doc 3 vs Doc 4: 0.063
